# A_01 · Ingesta de noticias: calidad medida e «impacto de la noticia» con FinBERT

**Carril A.** Autor: Daniel.

La ingesta de noticias del proyecto (`ingest/news.py` + `ingest/article_meta.py`) ya hace tres cosas sobre cada noticia seleccionada:

1. **URL real del medio** en lugar del enlace de redirección de Google News.
2. **Extracto breve** (`og:description`) cuando el feed solo trae el titular.
3. **Relevancia por ticker** (`relevance_score`), con los motivos de cada elección.

Este notebook hace dos cosas:

| Parte | Qué aporta | Modelos |
|---|---|---|
| **A · Medir la ingesta** (secciones 5-6) | Antes / después del enriquecimiento con noticias reales y la relevancia explicada | — |
| **B · Impacto de la noticia** (secciones 1-4 y 7-8) | Nuevo paso `ingest.impact`: tono positivo / negativo / neutral de cada noticia | **Claude Haiku** (traducción) → **FinBERT** (clasificación) |

El paso nuevo es **opcional** (`BRIEFER_FINBERT`), corre **en paralelo con el Agente Analista** (no alarga el briefing) y, si falla, el briefing sigue igual. No cambia ningún contrato.

**Cómo usarlo:** ejecutar las celdas en orden desde la carpeta `notebooks/`. El notebook **solo importa** el código del repo y no reescribe ningún fichero: el módulo vive en `src/briefer/ingest/sentiment.py`, el paso en la sección «3b» de `src/briefer/pipeline.py` y los tests en `tests/test_ingest_sentiment.py`. (La primera versión de este notebook generaba esos ficheros con `%%writefile`; se quitó para que volver a ejecutarlo no pise cambios posteriores.)

## 0 · Preparación

Sitúa el notebook en la raíz del repo (así las rutas relativas funcionan), añade `src/` al path y activa `autoreload` para que los cambios en los módulos se recojan sin reiniciar el kernel.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

print("Raíz del repo:", ROOT)
assert (ROOT / "src" / "briefer" / "pipeline.py").exists(), "Ejecuta el notebook desde la carpeta notebooks/ del repo"

Raíz del repo: c:\Users\danie\Desktop\MIAX - BME\Bloque 5 - Inteligencia Artificial Generativa\Multimodales\Taller_B5_T4_OscarRomeroQuincoces_FernandoDapenaTauste_DanielGarciaLopez\multimodal-market-briefer


## 1 · `sentiment.py`: «impacto de la noticia» con FinBERT

**Qué es FinBERT.** Un BERT ajustado con noticias financieras (`ProsusAI/finbert`, Hugging Face) que clasifica un texto en *positive / negative / neutral*. Es un **modelo abierto que corre en local** (CPU): suma un modelo especializado más a la orquestación, que es uno de los criterios de evaluación.

**Problema: solo entiende inglés**, y la mayoría de nuestras noticias están en español. Solución, encadenando dos modelos:

1. **Claude Haiku** traduce al inglés, en **una sola llamada** con salida estructurada (`_Translations`), el titular y el extracto de cada noticia que no esté ya en inglés.
2. **FinBERT** clasifica cada texto y devuelve la probabilidad de cada clase.

El texto que se clasifica es **titular + extracto**, así que aprovecha el extracto que añade la ingesta.

**Por qué «impacto de la noticia» y no «sentimiento del valor».** Etiquetar acciones como positivas o negativas puede leerse como una recomendación implícita según el reglamento de abuso de mercado (MAR, ver `docs/04`). Aquí se clasifica **el tono de la noticia**, sin agregarlo por ticker ni decir qué hacer con la acción.

**Robustez.**
- `torch` + `transformers` están en `requirements-local.txt` y el modelo (~440 MB) se descarga la primera vez. Si no están instalados, `news_impact` devuelve `{}` y el briefing sigue.
- Sin LLM real (modo demo) solo se clasifican las noticias en inglés; si la traducción falla, esas noticias se omiten.
- **Sin globales**, igual que `fetch_news`: las estadísticas de cada llamada van en `stats_out` y `impact_detail` las resume para la traza «Cómo se hizo».

In [ ]:
import inspect

from briefer.ingest import sentiment

print("Módulo:", Path(sentiment.__file__).relative_to(ROOT))
print("Modelo:", sentiment.FINBERT_MODEL, "· máx. caracteres por noticia:", sentiment.MAX_TEXT_CHARS)
print()
# Función principal (el resto del módulo: translate_to_english, classify, to_impact, impact_detail)
print(inspect.getsource(sentiment.news_impact))

## 2 · Conectar el paso al pipeline y a la configuración

La integración ya está en el repo (en tres ficheros); la celda siguiente solo la **lee** y la muestra:

- **`config.py`**: interruptor `briefer_finbert` (apagado por defecto).
- **`.env.example`**: documenta `BRIEFER_FINBERT`.
- **`pipeline.py`**: paso opcional **`ingest.impact`** (sección «3b»), solo con noticias reales:
  - Se lanza en un hilo **justo antes del Agente Analista** y corre **en paralelo** con él, porque el Analista no necesita su resultado. Así los ~10 s de traducción + FinBERT no alargan el briefing.
  - Al terminar el Analista se espera al hilo y se añade su `StepMetric` (latencia, coste de Haiku medido con `_MeteredLLM` y resumen ▲ ▼ ●). Si el Analista falla, el hilo también se espera (bloque `finally`), así no quedan hilos huérfanos.
  - El resultado por noticia se guarda en `news_impact.json`, en la carpeta del briefing, **fuera del contrato `Briefing`**, para poder enseñarlo en la UI más adelante sin cambiar contratos.
  - Va dentro de `_optional_step`: un fallo queda marcado en la traza y el briefing continúa.

In [ ]:
from briefer.config import Settings

pipeline_src = (ROOT / "src" / "briefer" / "pipeline.py").read_text(encoding="utf-8")
start = pipeline_src.index("    # 3b.")
end = pipeline_src.index("    def _analyze_with", start)
print(pipeline_src[start:end])
print("BRIEFER_FINBERT por defecto:", Settings(_env_file=None).briefer_finbert)

## 3 · Tests sin red ni `torch`

Están en `tests/test_ingest_sentiment.py`. FinBERT se sustituye por un clasificador falso y Haiku por un LLM falso que «traduce». Comprueban que:

- el texto clasificado es titular + extracto, recortado a 400 caracteres;
- solo se traducen las noticias que no están en inglés, en **una** llamada por lote, y sin LLM real, con el mock o si la API falla, se clasifican solo las inglesas;
- las etiquetas y probabilidades llegan bien y `impact_detail` resume ▲ ▼ ●;
- si FinBERT está desactivado, no instalado o falla, todo sigue funcionando;
- **en el pipeline**: el paso corre **en paralelo** con el Analista (los dos se esperan en una barrera; si fueran en serie, la barrera agotaría su tiempo y el test fallaría), su métrica aparece justo después de la del Analista y escribe `news_impact.json`;
- sin el interruptor o con noticias de ejemplo, el paso no se ejecuta, y si falla, el briefing sale igual.

Ejecutamos los tests nuevos y después **toda la batería**, para asegurar que no se ha roto nada del resto del equipo. Deben pasar todos.

In [5]:
!{sys.executable} -m pytest -q tests/test_ingest_sentiment.py
!{sys.executable} -m pytest -q

.........                                                                [100%]
9 passed in 1.18s
........................................................................ [  7%]
........................................................................ [ 14%]
........................................................................ [ 21%]
........................................................................ [ 29%]
........................................................................ [ 36%]
........................................................................ [ 43%]
........................................................................ [ 50%]
........................................................................ [ 58%]
........................................................................ [ 65%]
........................................................................ [ 72%]
........................................................................ [ 79%]
......................

## 4 · Calidad de la ingesta: antes y después del enriquecimiento

A partir de aquí se usa la red. Se descargan las noticias reales dos veces con la ingesta del proyecto (`fetch_news`):

- **sin enriquecer** (`enrich=False`): lo que dan los feeds tal cual;
- **enriquecidas** (`enrich=True`, sin caché): URL del medio y extracto (`og:description`).

Con `stats_out` se recogen las estadísticas de cada llamada, y `format_news_stats` las resume como en la traza «Cómo se hizo».

In [6]:
import pandas as pd

from briefer.ingest import news

pd.set_option("display.max_colwidth", 90)
TICKERS = ["SAN.MC", "ITX.MC", "IBE.MC", "AAPL", "NVDA", "^IBEX", "^GSPC"]


def resumen(items):
    total = len(items)
    con_extracto = sum(bool(n.summary) for n in items)
    google = sum("news.google.com" in n.url for n in items)
    return {"noticias": total, "con extracto": f"{con_extracto} ({round(100 * con_extracto / max(total, 1))} %)",
            "enlaces de Google News": google}


stats_sin, stats_con = {}, {}
sin_enriquecer = news.fetch_news(TICKERS, max_items=20, enrich=False, stats_out=stats_sin)
enriquecidas = news.fetch_news(TICKERS, max_items=20, enrich=True, use_cache=False, stats_out=stats_con)

display(pd.DataFrame({"sin enriquecer": resumen(sin_enriquecer), "enriquecidas": resumen(enriquecidas)}))
print("Traza «Cómo se hizo»:", (news.format_news_stats(stats_con) or "")[:400], "…")

19:02:19 INFO    briefer.ingest.news: Noticias: 357 obtenidas, 8 casi duplicadas, 151 fuera de ventana, 20 seleccionadas (14 con extracto; 30 fuentes, 0 fallidas)
19:02:22 INFO    briefer.ingest.news: Enriquecimiento de noticias: 7 candidatas, 7 URL resueltas, 5 extractos, 0 sin tiempo (1.0 s)
19:02:22 INFO    briefer.ingest.news: Noticias: 357 obtenidas, 9 casi duplicadas, 149 fuera de ventana, 20 seleccionadas (18 con extracto; 30 fuentes, 0 fallidas)


,sin enriquecer,enriquecidas
noticias,20,20
con extracto,14 (70 %),18 (90 %)
enlaces de Google News,6,0


Traza «Cómo se hizo»: 30 fuentes (0 fallidas, 0 de caché) en 2.0 s: Google News 140, Bing News 81, Yahoo RSS 106, prensa 30, yfinance 0 · 20 seleccionadas (18 con extracto) · extracto en el 90 % · descartadas: 12 fichas de cotización, 27 duplicadas, 9 casi duplicadas, 149 fuera de ventana (48 h), 140 por cupo de relevancia · extractos: 7 URL resueltas, 5 nuevos (0 sin tiempo) · relevancia: ^IBEX 6.5 «Los valores del Ib …


In [7]:
pd.DataFrame(
    [{"medio": n.source, "titular": n.title[:70], "url": n.url[:60], "extracto": n.summary[:120]} for n in enriquecidas]
)

,medio,titular,url,extracto
0,Mercados // expansion,Los valores del Ibex más alcistas por el 'efecto Brasil',https://www.expansion.com/mercados/2026/10/05/6ac36942e5fdea,El Ibex ha sido el mejor índice bursátil de la jornada en Europa. Este impulso ha sido...
1,CapitalMadrid,"Banco Santander, Mapfre y Telefónica impactan en Bolsa los resultados",https://www.capitalmadrid.com/2026/10/5/72521/banco-santande,"Telefónica, que avanza un 2,5% en bolsa, es el otro valor español donde los inversores..."
2,Mercados // expansion,Los hedge fund cuantitativos obtienen grandes ganancias con las ventas,https://www.expansion.com/economia/financial-times/2026/10/0,Las carteras que siguen tendencias han aprovechado el fuerte repunte de los rendimient...
3,Europa Press - Economía Finanzas,"El Ibex 35 acelera y roza los 19.300 puntos (+1,12%), tras la convocat",https://www.europapress.es/economia/bolsa-00348/noticia-ibex,"El Ibex 35 ha finalizado la sesión de este lunes con un repunte del 1,12%, hasta situa..."
4,TIKR.com,Wall Street Lenders Challenge Nvidia Valuation on Advanced AI Hardware,https://www.tikr.com/es/blog/wall-street-lenders-challenge-n,Nvidia stock in focus as Wall Street lenders question how long AI chips hold value and...
5,El Boletín,Wall Street abre mixto con el S&P 500 y el Nasdaq al alza pese a la pr,https://www.elboletin.com/wall-street-abre-mixto-con-el-sp-5,Wall Street abre mixto sin dirección clara. Descubre el impacto del débil informe de e...
6,Bloomberg Línea,S&P 500 avanza pese a presiones en Europa y el real lidera las monedas,https://www.bloomberglinea.com/mercados/sp-500-avanza-pese-a,"Los problemas fiscales y políticos de Europa elevan la cautela global, mientras las me..."
7,Yahoo Finanzas,El S&P 500 subió +0.5% en septiembre; esta estrategia de IA ganó +13.4,https://es-us.finanzas.yahoo.com/noticias/s-p-500-subi%C3%B3,Mientras los índices del mercado se movieron lateralmente en septiembre —dejando al S&...
8,estrategiasdeinversion.com,"NVIDIA, en máximos: ¿lista para alcanzar los 6 billones de dólares de",https://www.estrategiasdeinversion.com/actualidad/noticias/b,Los mercados de opciones muestran una probabilidad del 50% a que NVIDIA alcance los 6 ...
9,Expansión,"El Ibex, condicionado por el euro, la deuda y la política",https://www.expansion.com/mercados/cronica-bolsa/2026/10/05/,La semana comienza con varios frentes abiertos en los mercados. El euro cae a mínimos ...


## 5 · Relevancia explicada

Para cada noticia elegida, la ingesta guarda el ticker que la justifica, su puntuación (`relevance_score`) y los **motivos** (mención en el titular, horas desde su publicación, idioma, extracto…). Las del principio son las que el Analista recibe primero.

In [8]:
seleccion = stats_con["quality"]["selection"]
pd.DataFrame(
    [
        {
            "ticker": s.get("ticker") or "general",
            "puntuación": s.get("score"),
            "motivos": ", ".join(s.get("reasons") or []),
            "extracto": "sí" if s.get("has_summary") else "no",
            "titular": str(s.get("title", ""))[:80],
        }
        for s in seleccion
    ]
)

,ticker,puntuación,motivos,extracto,titular
0,^IBEX,6.466,"titular, 0 h, es, extracto",sí,Los valores del Ibex más alcistas por el 'efecto Brasil'
1,SAN.MC,6.457,"titular, 1 h, es, extracto",sí,"Banco Santander, Mapfre y Telefónica impactan en Bolsa los resultados de Brasil"
2,general,NaN,contexto general,sí,Los hedge fund cuantitativos obtienen grandes ganancias con las ventas de bonos
3,^IBEX,6.423,"titular, 1 h, es, extracto",sí,"El Ibex 35 acelera y roza los 19.300 puntos (+1,12%), tras la convocatoria elect"
4,NVDA,5.897,"titular, 1 h, es",sí,Wall Street Lenders Challenge Nvidia Valuation on Advanced AI Hardware Assets
5,^GSPC,5.762,"titular, 3 h, es",sí,Wall Street abre mixto con el S&P 500 y el Nasdaq al alza pese a la presión de l
6,^GSPC,5.746,"titular, 3 h, es",sí,S&P 500 avanza pese a presiones en Europa y el real lidera las monedas tras vent
7,^GSPC,5.709,"titular, 4 h, es",sí,El S&P 500 subió +0.5% en septiembre; esta estrategia de IA ganó +13.4%
8,NVDA,6.190,"titular, 4 h, es, extracto",sí,"NVIDIA, en máximos: ¿lista para alcanzar los 6 billones de dólares de valor en B"
9,^IBEX,6.176,"titular, 4 h, es, extracto",sí,"El Ibex, condicionado por el euro, la deuda y la política"


## 6 · FinBERT: impacto de cada noticia

**Requisito (una sola vez):** instalar las dependencias locales. En Windows conviene instalar antes `torch` para CPU, que pesa mucho menos que la versión con CUDA:

```
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements-local.txt
```

La celda usa el LLM barato del proyecto (Haiku) para traducir, así que necesita `ANTHROPIC_API_KEY` en el `.env`. Sin clave, solo se clasifican las noticias que ya están en inglés. La primera ejecución descarga el modelo (~440 MB) y tarda algo más.

In [9]:
from briefer.config import get_settings
from briefer.ingest import sentiment
from briefer.providers import registry

print("FinBERT disponible:", sentiment.finbert_available())
llm = registry.get_llm(get_settings(), cheap=True)
print("LLM para traducir:", llm.provider_name, llm.model)

stats_impacto = {}
impactos = sentiment.news_impact(enriquecidas, llm, stats_out=stats_impacto)
print(sentiment.impact_detail(stats_impacto), f"· {stats_impacto.get('total_s')} s")

pd.DataFrame(
    [
        {
            "impacto": impactos[n.id].impact,
            "prob.": impactos[n.id].score,
            "traducida": impactos[n.id].translated,
            "titular": n.title[:70],
            "texto que ve FinBERT": impactos[n.id].text_en[:90],
        }
        for n in enriquecidas
        if n.id in impactos
    ]
)

FinBERT disponible: True
LLM para traducir: anthropic claude-haiku-4-5-20251001


c:\dev\10k-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 48776.62it/s]
19:02:45 INFO    briefer.ingest.sentiment: FinBERT cargado en 1.5 s


FinBERT: 20 noticias (20 traducidas con Haiku) · ▲ 10 · ▼ 6 · ● 4 · 23.53 s


,impacto,prob.,traducida,titular,texto que ve FinBERT
0,positivo,0.9470,True,Los valores del Ibex más alcistas por el 'efecto Brasil',Ibex's most bullish values due to the 'Brazil effect'. The Ibex has been the best stoc...
1,neutral,0.8769,True,"Banco Santander, Mapfre y Telefónica impactan en Bolsa los resultados","Banco Santander, Mapfre and Telefónica impact the stock market with Brazil results. Te..."
2,positivo,0.9525,True,Los hedge fund cuantitativos obtienen grandes ganancias con las ventas,Quantitative hedge funds gain significantly from global bond sales. Trend-following po...
3,positivo,0.9432,True,"El Ibex 35 acelera y roza los 19.300 puntos (+1,12%), tras la convocat","Ibex 35 accelerates and nears 19,300 points (+1.12%), following the electoral call. Th..."
4,negativo,0.9431,True,Wall Street Lenders Challenge Nvidia Valuation on Advanced AI Hardware,Wall Street Lenders Challenge Nvidia Valuation on Advanced AI Hardware Assets. Nvidia ...
5,negativo,0.8889,True,Wall Street abre mixto con el S&P 500 y el Nasdaq al alza pese a la pr,Wall Street opens mixed with the S&P 500 and Nasdaq up despite bond pressure. Wall Str...
6,positivo,0.6827,True,S&P 500 avanza pese a presiones en Europa y el real lidera las monedas,S&P 500 advances despite pressures in Europe and the real leads currencies following B...
7,positivo,0.9308,True,El S&P 500 subió +0.5% en septiembre; esta estrategia de IA ganó +13.4,S&P 500 rose +0.5% in September; this AI strategy gained +13.4%. While market indices ...
8,positivo,0.9052,True,"NVIDIA, en máximos: ¿lista para alcanzar los 6 billones de dólares de",NVIDIA at all-time highs: ready to reach $6 trillion market capitalization? Options ma...
9,negativo,0.9278,True,"El Ibex, condicionado por el euro, la deuda y la política","Ibex, conditioned by the euro, debt and politics. The week begins with several open fr..."


## 7 · Briefing real completo: el paso nuevo en la traza

Lanza un briefing real de punta a punta (necesita las claves del `.env`) con FinBERT activado solo para esta ejecución, y muestra las métricas: son las mismas filas que aparecen en la pestaña «Cómo se hizo» de la app. Fíjate en que `ingest.impact` aparece justo después de `agents.analyst`: ha corrido en paralelo con él.

Para activar FinBERT también en la app, poner `BRIEFER_FINBERT=true` en el `.env`.

In [10]:
import json

from briefer import pipeline
from briefer.config import reset_settings_cache

os.environ["BRIEFER_FINBERT"] = "true"
reset_settings_cache()

briefing = pipeline.run_briefing(["SAN.MC", "ITX.MC", "AAPL"], mode="real")
print(f"Briefing {briefing.id} · coste total ≈ {sum(m.est_cost_eur for m in briefing.metrics):.3f} €")

display(
    pd.DataFrame([m.model_dump() for m in briefing.metrics])[["step", "provider", "latency_s", "est_cost_eur", "error", "detail"]]
)

impact_file = get_settings().output_path / briefing.id / "news_impact.json"
print("news_impact.json:", impact_file.exists())
pd.DataFrame(json.loads(impact_file.read_text(encoding="utf-8")))[["news_id", "impact", "score", "translated"]] if impact_file.exists() else None

19:02:46 WARNING briefer.providers.registry: stt: usando mock en lugar de 'whisper_api' (Falta OPENAI_API_KEY en .env para stt='whisper_api')
19:02:46 INFO    briefer.pipeline: (1/6) Recopilando noticias y precios…
19:02:46 INFO    briefer.pipeline: ingest.prices [yfinance/-] 0.01s 0.0000€
19:02:46 INFO    briefer.ingest.news: Enriquecimiento de noticias: 8 candidatas, 8 URL resueltas, 6 extractos, 0 sin tiempo (0.0 s)
19:02:46 INFO    briefer.ingest.news: Noticias: 259 obtenidas, 6 casi duplicadas, 98 fuera de ventana, 20 seleccionadas (18 con extracto; 22 fuentes, 0 fallidas)
19:02:46 INFO    briefer.pipeline: ingest.news [yfinance+rss/-] 0.09s 0.0000€
19:02:46 INFO    briefer.pipeline: ingest.tickers [local/-] 0.00s 0.0000€
19:02:46 INFO    briefer.pipeline: (2/6) Analizando el mercado…
19:02:57 INFO    briefer.pipeline: agents.analyst [anthropic/claude-sonnet-5-5] 10.80s 0.0239€
19:03:01 INFO    briefer.pipeline: ingest.impact [finbert+haiku/ProsusAI/finbert] 15.07s 0.0078€
19:03:0

Briefing 20261005-190246-098c85 · coste total ≈ 0.039 €


,step,provider,latency_s,est_cost_eur,error,detail
0,ingest.news,yfinance+rss,0.0934,0.000000,None,"22 fuentes (0 fallidas, 22 de caché) en 0.0 s: Google News 100, Bing News 57, Yahoo RS..."
1,ingest.prices,yfinance,0.0062,0.000000,None,None
2,ingest.tickers,local,0.0035,0.000000,None,20 de 20 noticias relevantes (8 de índices de contexto)
3,agents.analyst,anthropic,10.7993,0.023906,None,grounding: todas las cifras trazables
4,ingest.impact,finbert+haiku,15.0689,0.007777,None,FinBERT: 20 noticias (20 traducidas con Haiku) · ▲ 11 · ▼ 5 · ● 4
5,agents.scriptwriter,anthropic,9.7365,0.007270,None,guion: cifras trazables al análisis · guion: 1 frase(s) con causa no matizada · guion:...
6,media.podcast,edge,7.0099,0.000000,None,None
7,media.transcript,local,0.0009,0.000000,None,None
8,media.charts,matplotlib,0.3285,0.000000,None,None
9,storage.save,local,0.0018,0.000000,None,None


news_impact.json: True


,news_id,impact,score,translated
0,n-b965364207e1c893,positivo,0.9437,True
1,n-af79d49cab65204f,neutral,0.7948,True
2,n-a35093b0a3ec6447,positivo,0.9418,True
3,n-b51d9c7facaad8c8,positivo,0.9468,True
4,n-4526815132c096b5,negativo,0.8819,True
5,n-828db82e89f9ad2b,positivo,0.6538,True
6,n-010a63f0bfb68c60,positivo,0.9436,True
7,n-6acfedf19ebe138f,positivo,0.8131,True
8,n-8e7f8d7f0bcdedbf,negativo,0.9371,True
9,n-c7a19cbf11c5237d,negativo,0.9661,True


## 8 · Conclusiones y siguientes pasos

**Medido el 05-oct-2026** con noticias reales:

- **Calidad de la ingesta** (sección 4): con el enriquecimiento, las noticias con extracto pasan de **14/20 (70 %) a 18/20 (90 %)** y los enlaces de Google News, de **6 a 0** (todos resueltos al medio). En una medición anterior, solo con Google News: de 6 % a 88 %.
- **FinBERT** (sección 6): 20 noticias traducidas con Haiku y clasificadas (▲ 10 · ▼ 6 · ● 4) en ~24 s, con la mayoría de etiquetas coherentes con los titulares. Por ejemplo, «Wall Street cuestiona la valoración de Nvidia» → negativa (0,94); «Jefferies apuesta por Inditex» → positiva (0,95).
- **Coste y tiempo**: el paso `ingest.impact` cuesta ~0,008 € por briefing (la traducción con Haiku; FinBERT es local y gratis). Corre en paralelo con el Analista (15,1 s frente a 10,8 s), así que solo añade unos 4 s al briefing. Briefing completo: ~0,039 €.
- **Limitaciones observadas**: FinBERT lee el tono literal y a veces se equivoca con las metáforas. «Inditex mira por el retrovisor a Shein y ya es seis veces más rentable» sale negativa (por *rearview mirror* en la traducción). Por eso se presenta como **segunda opinión** junto al sentimiento del Analista, no como verdad.
- **Orquestación**: suma un modelo abierto local y un encadenamiento más (Haiku → FinBERT) al flujo multimodal.

**Pendiente (a acordar con el carril B):**
1. ~~Enseñar el impacto de cada noticia en la UI~~ **Hecho**: en «Puntos clave», cada fuente con impacto lleva la etiqueta «impacto de la noticia: ▲ positiva · FinBERT» (lee `news_impact.json` con `storage.load_news_impact`), y `export_briefing` copia ese fichero.
2. Decidir si se pasa al Analista como «segunda opinión». Requiere añadir un campo al contrato (`MarketContext`).